In [14]:
import json

with open("../data/ocr_pages.json", "r", encoding="utf-8") as f:
    pages = json.load(f)

print("Number of pages:", len(pages))

Number of pages: 32


In [15]:
page = pages[2]

print("Page:", page["page"])
print()

for line in page["lines"]:
    print(line)

Page: 3

لا عمل إلا بنية
بسم الله الرحمن الرحيم
لا عمل إا بنية
عن أمير المؤمنين آبي حفص عمر بن
الخطاب رضي الله تعالى عنه قال ٠ سمعت رسول
الله صلى الله عليه وسلم يقول ٠ ) إنًمًا الأعمال بالنًيًات وإنًما لكلد
امرئ ما نوًى فمن كانت هجرته إلى اللًه ورسوله
فهجرته إلى اللًه ورسوله ومن كانت هجرته لدنيًا
يصيبها أو امرأة ينكحهًا ؛ فهجرته إل مًا هاجر إليه (
رواه إماما المحدثين أبو عبد اللًه محمد بن إسماعيل
ابن إبراهيم بن المغيرقة بن بردزبه البخاري م وأبو الحسين
مسلم بن الحجًاج بن مسلم القشيريً النًيسابوريً في
صحيحيهما اللذين هما أصح الكتب المصنًفة (١)
(١ أخرجه البخاري في بدء الوحي )ا( ومسلم في الإمارة )١٥٥(
قوله ٠ ل النيات ( أي القصد وعزم القلب على الفعل


In [16]:
import json
import os

from openai import OpenAI
from dotenv import load_dotenv

In [17]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    raise ValueError("OPENROUTER_API_KEY is not set.")

print("API key loaded successfully.")

API key loaded successfully.


In [18]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)
MODEL_NAME = "inclusionai/ling-3.0-flash-vl:free"

In [19]:
SYSTEM_PROMPT = """
You are an OCR correction assistant for a classical Arabic Islamic book
(Al-Arba'een Al-Nawawiyyah — a collection of 40 hadiths with commentary).

Your ONLY task is to fix OCR recognition errors. You are not an editor,
not a scholar, and not allowed to "improve" or "correct" the religious
text based on what you remember from other sources.

CONTEXT — the input line may be one of:
- A hadith number marker (e.g. "١ –", "٢ –") — keep exactly as is.
- A section/chapter title (originally inside a decorative box).
- The matn (body) of a hadith — narrator chain + Prophetic wording.
- A footnote: explanation of a word (غريب الحديث) or a reference/takhrij
  (e.g. "[رواه البخاري ومسلم]", numbers in parentheses like "(١)").
Do not merge or reorder these; treat the input as a single line/segment
and fix only what's inside it.

STRICT RULES:
1. Preserve the original text and meaning exactly.
2. Do NOT summarize, rewrite, or paraphrase.
3. Do NOT add or remove information, words, or phrases.
4. Do NOT change the order of words or sentences.
5. Do NOT "correct" wording to match a version of the hadith you recall
   from memory. If the input differs from what you remember but reads
   as a plausible original sentence, LEAVE IT UNCHANGED — this may be
   a real variant wording, not an OCR error.
6. Preserve narrator names, chains of narration (isnad), book titles,
   hadith references, and all religious terminology exactly as spelled,
   unless clearly a letter-level OCR glitch (see examples below).
7. Diacritics (tashkeel): only fix a diacritic mark if it is obviously
   broken/garbled (e.g. a stray unrecognizable symbol in place of a
   harakah). Do NOT add diacritics that are missing — leaving a word
   undiacritized is always safer than guessing a diacritic.
8. Fix only clear, low-level OCR artifacts:
   - misrecognized/confused Arabic letters (e.g. ه/ة, ي/ى, ب/ت/ث)
   - missing or duplicated letters within a word
   - broken/merged word spacing
   - garbled punctuation
   - corrupted decorative glyphs — see examples below
9. If you are not confident something is an OCR error, leave it
   UNCHANGED. Never guess.
10. Output ONLY the corrected Arabic text — no explanations, no
    markdown, no quotation marks, no comments.

EXAMPLES OF KNOWN OCR GLITCHES IN THIS BOOK (apply the same pattern to
similar garbled tokens near the Prophet's name or at the top of a page,
even if the exact garbled letters differ):
- Garbled token appearing right after mentions of the Prophet (from a
  misread ﷺ ligature), e.g. "عطقه", "عقطته", "علقاهم" → "صلى الله عليه وسلم"
- A large garbled token at the very top of a hadith/page (from a
  misread Basmalah ornament), e.g. "س إلقوالثفز قليي" → "بسم الله الرحمن الرحيم"
- If you see a short, meaningless token in a position where a religious
  formula (صلى الله عليه وسلم / رضي الله عنه / تعالى) would normally
  appear, and it does not form a real Arabic word, treat it as a
  corrupted formula and restore the standard phrasing that fits the
  grammatical context — but only when no real alternative reading of
  the token as an actual word is plausible.
"""

In [23]:
def correct_page_with_llm(page_lines):
    page_text = "\n".join(page_lines)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": page_text
            }
        ],
        temperature=0
    )

    message = response.choices[0].message

    if message.content is None:
        print("LLM returned no text.")
        print("Response:")
        print(response)
        return None

    return message.content.strip()

In [21]:
test_page = pages[2]

corrected_text = correct_page_with_llm(test_page["lines"])

print(corrected_text)

لا عمل إلا بنية
بسم الله الرحمن الرحيم
لا عمل إلا بنية
عن أمير المؤمنين أبى حفص عمر بن
الخطاب رضي الله تعالى عنه قال ٠ سمعت رسول
الله صلى الله عليه وسلم يقول ٠ ( إنَّمَا الأعمال بالنِّيَّاتِ وإنَّمَا لكلِّ
امرئ ما نَوَى فمن كانت هجرته إلى اللَّهِ ورسوله
فهجرته إلى اللَّهِ ورسوله ومن كانت هجرته لدنيًا
يصيبها أو امرأة ينكحهًا ؛ فهجرته إلَّمَا هاجر إليه )
رواه إماما المحدثين أبو عبد اللَّهِ محمد بن إسماعيل
ابن إبراهيم بن المغيرقة بن بردزة البخاري م وأبو الحسين
مسلم بن الحَجَّاجِ بن مسلم القشيرِيِّ النَّيسَابُورِيِّ في
صحيحيهما اللذين هما أصح الكتب المُصَفَّفَةِ (١)
(١ أخرجه البخاري في بدء الوحي (ا) ومسلم في الإمارة (١٥٥)
قوله ٠ للنيات ( أي القصد وعزم القلب على الفعل


In [25]:
cleaned_pages = []

for page in pages[9:]:
    print(f"Processing page {page['page']}...")

    corrected_text = correct_page_with_llm(page["lines"])

    if corrected_text is None:
        print(f"Skipping page {page['page']} because LLM returned no text.")
        break

    cleaned_pages.append({
        "page": page["page"],
        "text": corrected_text
    })

print("Finished.")

Processing page 10...
Processing page 11...
Processing page 12...
Processing page 13...
Processing page 14...
Processing page 15...
Processing page 16...
Processing page 17...
Processing page 18...
Processing page 19...
Processing page 20...
Processing page 21...
Processing page 22...
Processing page 23...
Processing page 24...
Processing page 25...
Processing page 26...
Processing page 27...
Processing page 28...
Processing page 29...
Processing page 30...
Processing page 31...
Processing page 32...
Finished.


In [27]:
output_path = "../data/cleaned_pages.json"

# اقرأ البيانات الموجودة لو الملف موجود
if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        existing_pages = json.load(f)
else:
    existing_pages = []

# الصفحات الموجودة بالفعل
existing_page_numbers = {
    page["page"] for page in existing_pages
}

# أضف فقط الصفحات الجديدة
for page in cleaned_pages:
    if page["page"] not in existing_page_numbers:
        existing_pages.append(page)

# احفظ كل البيانات مرة أخرى
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        existing_pages,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Total pages saved: {len(existing_pages)}")
print(f"Saved to: {output_path}")

Total pages saved: 32
Saved to: ../data/cleaned_pages.json


In [1]:
import json
from pathlib import Path

INPUT_FILE = Path("../data/cleaned_pages.json")
OUTPUT_FILE = Path("../data/corrected_pages.json")


# ============================================================
# OCR CORRECTIONS
# Each correction is tied to a specific page.
# This is safer than global replacements.
# ============================================================

CORRECTIONS = {

    1: {
        "ألنجاين لثو ثيتنا": "الأربعين النووية",
        "لإمام أبو زكريا بن شرف النووي": "للإمام أبي زكريا يحيى بن شرف النووي",
        "خرج أحاديثه وشكح غربه": "خرج أحاديثه وشرح غريبه",
        "أحمد عبدا الرازق البكري": "أحمد عبد الرازق البكري",
        "ذو السارين": "ذو الساريين",
    },

    3: {
        "أبى حفص": "أبي حفص",
        "امرئ ما نَوَى": "امرئ ما نوى",
        "لدنيًا": "لدنيا",
        "ينكحهَا": "ينكحها",
        "فهجرته إلَّا مَنْ هاجر إليه": "فهجرته إلى ما هاجر إليه",
        "بردبة": "بردبة",
    },

    4: {
        "عن عمر قبه": "عن عمر رضي الله عنه",
        "لا يرى عليه آثر السفر": "لا يرى عليه أثر السفر",
        "ولا يغر فهه منًا أحد": "ولا يعرفه منا أحد",
        "فأسند ركبتيه إلى ركبتيه": "فأسند ركبتيه إلى ركبتيه",
        "تقيم الصلاة وتؤتي الزكاة": "تقيم الصلاة وتؤتي الزكاة",
        "تحج البيت إن استطعت إلى سبيلًا": "تحج البيت إن استطعت إليه سبيلًا",
        "فعجبنًا": "فعجبنا",
        "وملاًئكته": "وملائكته",
    },

    5: {
        "فانه جبريل": "فإنه جبريل",
        "من الأسئلة": "من السائل",
        "بني الإسلام على خمس": "بني الإسلام على خمس",
        "عن أبي عبد الرحمن عبد الله عمر بن": "عن أبي عبد الرحمن عبد الله بن عمر بن",
        "الثياب": "الثيب",
        "أيسيدتها": "أي سيدتها",
        "طويلا": "طويلًا",
    },

    6: {
        "الخلق والأجل والرذق": "الخلق والأجل والرزق",
        "يطمع خلقه": "يُجمع خلقه",
        "بأزع كلمات": "بأربع كلمات",
        "بكثرة رزقه": "بكتب رزقه",
        "أمل الجنة": "أهل الجنة",
        "أمل النار": "أهل النار",
        "فيدخله": "فيدخلها",
    },

    7: {
        "عن أم المؤمنين ; أم عبد الله": "عن أم المؤمنين أم عبد الله",
        "من ليس عليه أمرنا": "ليس عليه أمرنا",
        "من فهو رد": "منه فهو رد",
        "رواية سلم": "رواية مسلم",
        "أمرنًا": "أمرنا",
        "ال تعالى عنهما": "الله تعالى عنهما",
        "الآقضية": "الأقضية",
    },

    8: {
        "مضغةة": "مضغة",
        "وألا وهي القلب": "ألا وهي القلب",
        "النصح من أصول الإسلام": "النصح من أصول الإسلام",
        "أبي رقيعة": "أبي رقية",
        "تميم بن أوس الداري": "تميم بن أوس الداري",
        "ولأئمة المسلمين و عامتهم": "ولأئمة المسلمين وعامتهم",
    },

    9: {
        "ما أمرزتكم به": "ما أمرتكم به",
        "فأتوا من ما استطعتم": "فأتوا منه ما استطعتم",
        "فإنّمًا": "فإنما",
        "كثرة مسائلهم": "كثرة مسائلهم",
        "عن أبي هريرة عبد الرحمن بن صهر": "عن أبي هريرة عبد الرحمن بن صخر",
    },

    10: {
        "أعملوا صالحًا": "اعملوا صالحًا",
        "يطيل السفر": "يطيل السفر",
        "أشعثًا أغبرًا": "أشعث أغبر",
        "يمد يديه": "يمد يديه",
        "وغذي بالحرام": "وغذي بالحرام",
        "فأنى يستجاب له": "فأنى يستجاب له",
    },

    11: {
        "مالا يريبك": "ما لا يريبك",
        "الاشتفال بما يفيد": "الاشتغال بما يفيد",
        "مالا يعنيه": "ما لا يعنيه",
        "ما يطحب لنفسه": "ما يحب لنفسه",
        "ال تعالى عنه": "الله تعالى عنه",
    },

    12: {
        "الثياح الزاني": "الثيب الزاني",
        "والنفس بالنفس": "والنفس بالنفس",
        "والتارك لدينه": "والتارك لدينه",
        "وأني رسول الله": "وأني رسول الله",
    },

    13: {
        "ا تفضب ولك الجنة": "لا تغضب ولك الجنة",
        "لا تغضب": "لا تغضب",
        "أبي يعلي": "أبي يعلى",
        "شدّاد بن أوس": "شداد بن أوس",
        "فأحسنوا القتلة": "فأحسنوا القتلة",
        "وليحد أحدكم شفرته": "وليحد أحدكم شفرته",
    },

    14: {
        "معاذ بن جبل": "معاذ بن جبل",
        "اتبع السنة الحسنة": "وأتبع السيئة الحسنة تمحها",
        "خلق الناس بخلق حسن": "وخالق الناس بخلق حسن",
        "علقته يومًا": "ردفته يومًا",
        "إذا سألت فاسأل الله": "إذا سألت فاسأل الله",
    },

    15: {
        "وجفًت الصحف": "وجفت الصحف",
        "عبر الترمذي": "عند الترمذي",
        "اخفظ الله": "احفظ الله",
        "أذرك الناس": "أدرك الناس",
        "فاضنع ما شئت": "فاصنع ما شئت",
    },

    16: {
        "أبي عمرةً": "أبي عمرة",
        "قل لي في الإسلام قولًا": "قل لي في الإسلام قولًا",
        "أرأيت إذا صليت المكتوبات": "أرأيت إذا صليت المكتوبات",
        "أحللت الحلال وحرمت الحرام": "وأحللت الحلال وحرمت الحرام",
        "عنهمًا": "عنهما",
    },

    17: {
        "فضل الله قنت": "فضل الله",
        "الطهور شطر الإيمان": "الطهور شطر الإيمان",
        "والصبر ضياءً": "والصبر ضياء",
        "القرآن حججة": "القرآن حجة",
        "كل النماس": "كل الناس",
        "فضل الله حبلا": "فضل الله",
    },

    18: {
        "أهدكم يا عبادي": "يا عبادي كلكم",
        "أكسكم": "أكسكم",
        "قازاد ذلك": "فما زاد ذلك",
        "أخصيها لكم": "أحصيها لكم",
        "المخيط إذا أدخل البخر": "المخيط إذا أدخل البحر",
    },

    19: {
        "ذهب أفهل الدهور بالأجور": "ذهب أهل الدثور بالأجور",
        "بفضول أموالهم": "بفضول أموالهم",
        "وأمر بالمغروف": "وأمر بالمعروف",
        "ونهي عن منكر": "ونهي عن منكر",
        "أرأًيتم": "أرأيتم",
        "فى الحلال": "في الحلال",
        "أبي هريرةً": "أبي هريرة",
    },

    20: {
        "فتخمله عليها": "فتحمله عليها",
        "الصلاقة": "الصلاة",
        "ويمط الأذى": "ويميط الأذى",
        "له صدلك": "له صدرك",
    },

    21: {
        "حديث خسن": "حديث حسن",
        "بإسناد خسن": "بإسناد حسن",
        "أبي نجيح الأعرض": "أبي نجيح العرباض",
        "فسيرى اختلافًا": "فسيرى اختلافًا",
        "الهدين": "المهديين",
        "بالنوازل": "بالنواجذ",
    },

    22: {
        "يسشره الله": "يسره الله",
        "وصلاة الرجل في جوف الليل": "وصلاة الرجل في جوف الليل",
        "لو ئتًجافى": "تتجافى",
        "عن المضاجع": "عن المضاجع",
        "يعملون": "يعملون",
    },

    23: {
        "فلا تضيعوه": "فلا تضيعوها",
        "وحدودًا فلا تعتدوها": "وحدودًا فلا تعتدوها",
        "وحرام أشياء": "وحرّم أشياء",
        "رحمة لكم": "رحمة لكم",
        "ثكلتك": "ثكلتك",
    },

    24: {
        "٣٢ - عن أبي سعيد": "٣٢ - عن أبي سعيد",
        "الضرر ولا ضرار": "لا ضرر ولا ضرار",
        "مسندًا": "مسندًا",
    },

    25: {
        "أنة رسول الله": "أن رسول الله",
        "لادعى رجال أموال قوم": "لادعى رجال أموال قوم",
        "وبغضه في الصحيحين": "وبعضه في الصحيحين",
        "أبي سعيد الخمدري": "أبي سعيد الخدري",
        "عقلاقم": "صلى الله عليه وسلم",
        "من رأى منكم منكرًا": "من رأى منكم منكرًا",
    },

    26: {
        "ولا تناجشوا": "ولا تناجشوا",
        "ولا يبع بعضكم على بيع بعض": "ولا يبع بعضكم على بيع بعض",
        "بحسب امرئ": "بحسب امرئ",
        "أن يخقر أخاه": "أن يحقر أخاه",
        "التقوى ههنا": "التقوى ههنا",
        "عن أبي هريرة - رضي الله تعالى عنه عن": "عن أبي هريرة رضي الله تعالى عنه",
    },

    27: {
        "يسهل الله له به طريقًا": "سهّل الله له به طريقًا",
        "يثلون كتاب الله": "يتلون كتاب الله",
        "ومغشيتهم": "وغشيتهم",
        "والسسيئات": "والسيئات",
    },

    28: {
        "غشر حسنات": "عشر حسنات",
        "الشيئة": "السيئة",
        "التَّمَأ كيد": "تمام التأكيد",
        "ولم يؤكدها بكاملة": "ولم يؤكدها بكاملة",
    },

    29: {
        "قال رسول الله عقلتم": "قال رسول الله",
        "فقد آذنته باحرب": "فقد آذنته بالحرب",
        "ممًا افترضته": "مما افترضته",
        "لأغطينه": "لأعطينه",
        "إن الله تعالى تجاوز لي عن أممي": "إن الله تعالى تجاوز لي عن أمتي",
    },

    30: {
        "بمنكباً": "بمنكبه",
        "من صختك": "من صحتك",
        "لمرضك": "لمرضك",
        "اتباع شرع الله عبل عماد إيمان": "اتباع شرع الله أصل عماد الإيمان",
    },

    31: {
        "ألا يؤمن أحدكم حتى يكون هواه تبعًا لما جعل به":
            "لا يؤمن أحدكم حتى يكون هواه تبعًا لما جئت به",
        "سعة مغفرة الله قكجنل": "سعة مغفرة الله عز وجل",
        "سعة مغفرة الله عخبل": "سعة مغفرة الله عز وجل",
        "خطاياً": "خطايا",
        "بقرابها مغفرة": "بقرابها مغفرة",
    },

}


# ============================================================
# APPLY CORRECTIONS
# ============================================================

def apply_corrections(text, corrections):
    """
    Apply exact phrase replacements.
    Longest phrases are replaced first to reduce
    accidental partial replacements.
    """
    changes = []

    for wrong, correct in sorted(
        corrections.items(),
        key=lambda x: len(x[0]),
        reverse=True
    ):
        if wrong in text:
            text = text.replace(wrong, correct)
            changes.append({
                "wrong": wrong,
                "correct": correct
            })

    return text, changes


# ============================================================
# LOAD
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    pages = json.load(f)


# ============================================================
# PROCESS
# ============================================================

all_changes = []

for page_data in pages:

    page_number = page_data["page"]
    original_text = page_data["text"]

    page_corrections = CORRECTIONS.get(page_number, {})

    corrected_text, changes = apply_corrections(
        original_text,
        page_corrections
    )

    page_data["text"] = corrected_text

    if changes:
        all_changes.append({
            "page": page_number,
            "changes": changes
        })


# ============================================================
# SAVE CORRECTED JSON
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        pages,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# PRINT REPORT
# ============================================================

print("=" * 60)
print("OCR CORRECTION FINISHED")
print("=" * 60)

print(f"Input : {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")

print(f"\nPages processed: {len(pages)}")
print(f"Pages modified : {len(all_changes)}")

total_changes = sum(
    len(page["changes"])
    for page in all_changes
)

print(f"Total replacements: {total_changes}")

print("\nChanges:")
print("-" * 60)

for page in all_changes:
    print(f"\nPage {page['page']}:")

    for change in page["changes"]:
        print(f"  ❌ {change['wrong']}")
        print(f"  ✅ {change['correct']}")

print("\nDone.")

OCR CORRECTION FINISHED
Input : ..\data\cleaned_pages.json
Output: ..\data\corrected_pages.json

Pages processed: 32
Pages modified : 29
Total replacements: 135

Changes:
------------------------------------------------------------

Page 1:
  ❌ لإمام أبو زكريا بن شرف النووي
  ✅ للإمام أبي زكريا يحيى بن شرف النووي
  ❌ أحمد عبدا الرازق البكري
  ✅ أحمد عبد الرازق البكري
  ❌ خرج أحاديثه وشكح غربه
  ✅ خرج أحاديثه وشرح غريبه
  ❌ ألنجاين لثو ثيتنا
  ✅ الأربعين النووية
  ❌ ذو السارين
  ✅ ذو الساريين

Page 3:
  ❌ فهجرته إلَّا مَنْ هاجر إليه
  ✅ فهجرته إلى ما هاجر إليه
  ❌ امرئ ما نَوَى
  ✅ امرئ ما نوى
  ❌ أبى حفص
  ✅ أبي حفص
  ❌ ينكحهَا
  ✅ ينكحها
  ❌ لدنيًا
  ✅ لدنيا
  ❌ بردبة
  ✅ بردبة

Page 4:
  ❌ تحج البيت إن استطعت إلى سبيلًا
  ✅ تحج البيت إن استطعت إليه سبيلًا
  ❌ تقيم الصلاة وتؤتي الزكاة
  ✅ تقيم الصلاة وتؤتي الزكاة
  ❌ فأسند ركبتيه إلى ركبتيه
  ✅ فأسند ركبتيه إلى ركبتيه
  ❌ ولا يغر فهه منًا أحد
  ✅ ولا يعرفه منا أحد
  ❌ عن عمر قبه
  ✅ عن عمر رضي الله عنه
  ❌ وملاًئكته
  ✅ وملائكته
  ❌ ف

In [ ]:
import json
import re

# إزالة التشكيل من بيانات الأحاديث النهائية (تحسين الاسترجاع)
TASHKEEL = "".join(chr(c) for c in list(range(0x064B, 0x0660)) + [0x0670])
TASHKEEL_RE = re.compile("[" + TASHKEEL + "]")

input_path = "../data/hadith_extraction_clean.json"

STR_FIELDS = ["id_label", "title", "narrator", "hadith_text",
              "source", "sharh", "rawi_bio", "page_title"]

with open(input_path, "r", encoding="utf-8") as f:
    hadiths = json.load(f)

fields_updated = 0
for h in hadiths:
    for field in STR_FIELDS:
        val = h.get(field)
        if isinstance(val, str) and val:
            new_val = re.sub(r"\s+", " ", TASHKEEL_RE.sub("", val)).strip()
            if new_val != val:
                fields_updated += 1
            h[field] = new_val

with open(input_path, "w", encoding="utf-8") as f:
    json.dump(hadiths, f, ensure_ascii=False, indent=2)

print(f"Hadiths processed: {len(hadiths)}")
print(f"Fields updated: {fields_updated}")
print(f"Saved to: {input_path}")
print()
print("Sample (hadith 1 narrator):")
print(hadiths[0]["narrator"])
